In [4]:
"""Cover fractions and 20% labels for every 1000, 500 and 250 px tile.

Reads the "tiles" and "rhodo" layers from labelling.gpkg and writes
tiles_1000px.gpkg, tiles_500px.gpkg and tiles_250px.gpkg.

Usage: python cover_fractions.py labelling.gpkg
Close QGIS (or save all edits) before running.
"""
import sys

import geopandas as gpd
from shapely.geometry import box

LABELLING_PATH = "../Grids/labelling.gpkg"
COVER_THRESHOLD = 0.20                   # cover >= 20% -> positive
LABELLED_STATUSES = ["done", "unsure"]   # "skip" and unlabelled tiles are left out
SIZES = [(1000, 1), (500, 2), (250, 4)]  # (tile size in px, splits per side)

tiles = gpd.read_file(LABELLING_PATH, layer="tiles")
rhododendron_outlines = gpd.read_file(LABELLING_PATH, layer="rhododendron_outlines")
rhododendron_outlines["geometry"] = rhododendron_outlines.make_valid()
rhodo_area = rhododendron_outlines.union_all()           # merge overlaps so no area counts twice

labelled = tiles[tiles["status"].isin(LABELLED_STATUSES)]
print(f"{len(labelled)} labelled tiles, {(tiles['status'] == 'skip').sum()} skipped, "
      f"{tiles['status'].isna().sum()} with no status")

keep = [c for c in tiles.columns if c != "geometry"]
rows = {size: [] for size, _ in SIZES}
for tile in labelled.itertuples(index=False):
    min_x, min_y, max_x, max_y = tile.geometry.bounds
    rhodo_in_tile = rhodo_area.intersection(tile.geometry)   # clip once per tile
    for size, splits in SIZES:
        step_x, step_y = (max_x - min_x) / splits, (max_y - min_y) / splits
        for sub_row in range(splits):                         # row 0 = top
            for sub_col in range(splits):
                sub = box(min_x + sub_col * step_x, max_y - (sub_row + 1) * step_y,
                          min_x + (sub_col + 1) * step_x, max_y - sub_row * step_y)
                cover = rhodo_in_tile.intersection(sub).area / sub.area
                label = int(cover >= COVER_THRESHOLD)
                rows[size].append({**{c: getattr(tile, c) for c in keep},
                                   "tile_size_px": size, "sub_row": sub_row, "sub_col": sub_col,
                                   "cover_fraction": round(cover, 4), "label": label,
                                   "class": "positive" if label else "negative",
                                   "geometry": sub})

for size, _ in SIZES:
    out = gpd.GeoDataFrame(rows[size], geometry="geometry", crs=tiles.crs)
    path = f"../Grids/tiles_{size}px.gpkg"
    out.to_file(path, layer=f"tiles_{size}px", driver="GPKG")
    print(f"{path}: {len(out)} tiles, {out['label'].sum()} positive "
          f"({out['label'].mean():.1%}), mean cover {out['cover_fraction'].mean():.3f}")

500 labelled tiles, 0 skipped, 4681 with no status
../Grids/tiles_1000px.gpkg: 500 tiles, 49 positive (9.8%), mean cover 0.059
../Grids/tiles_500px.gpkg: 2000 tiles, 174 positive (8.7%), mean cover 0.059
../Grids/tiles_250px.gpkg: 8000 tiles, 604 positive (7.5%), mean cover 0.059
